# COMP 4330SEF — Group 5: LightGCN Cold-Start Study (`Notebook-Hydrogen`)

This notebook implements the end-to-end **MovieLens 1M (`ml-1m`)** and **66-user pilot** workflow from the `4330SEF` codebase guideline:

1. **Environment & Repo Check** — verify GPU/CUDA, install `recbole` + `torch`, and locate `config/base.yaml`.
2. **Dataset & Cold-Item Slice Verification** — inspect `ml1m_split_stats.json`, `ml1m_cold_item_slices.json` (372 cold items, 28 zero-edge items), and `ml1m_genre_matrix.npz` (`[3629, 18]`).
3. **Fast Pilot Self-Test (`python/rec`)** — run the 8-arm NumPy pilot check (`selftest.py`) and visualize pilot Recall@10.
4. **Official `ml-1m` Baselines (RecBole)** — run `LightGCN`, `BPR`, `ItemKNN`, `NGCF`, `NeuMF`, and `SASRec` across seeds `2020`, `2021`, `2022`.
5. **Controlled Extensions** — run `LightGCNAdaptive` (learned softmax layer weights) and `LightGCNSide` (wired `[3629, 18]` genre matrix).
6. **Cold-Item Slice Evaluation & `results/runs.csv` Export** — invoke `scripts/cold_start_report.py` on saved checkpoints and plot final `ml-1m` comparisons.

## 1. Environment & Repository Setup

In [ ]:
# %% Environment check & optional dependency install (uncomment pip in fresh Colab/venv)
# !pip install -q recbole torch numpy pandas matplotlib pyyaml

import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config" / "base.yaml").exists() and (ROOT.parent / "config" / "base.yaml").exists():
    ROOT = ROOT.parent

for p in (str(ROOT), str(ROOT / "python")):
    if p not in sys.path:
        sys.path.insert(0, p)

print("Repo root:", ROOT)
try:
    import torch
    print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch not installed in this kernel (pilot numpy cells will still run).")

## 2. Verify `ml-1m` Split Stats, Cold-Item Slices & Genre Matrix

In [ ]:
# %% Inspect verified ml-1m split stats, cold-item slices, and genre matrix
import numpy as np

split_stats = json.loads((ROOT / "results" / "ml1m_split_stats.json").read_text(encoding="utf-8"))
slices = json.loads((ROOT / "results" / "ml1m_cold_item_slices.json").read_text(encoding="utf-8"))
genre_meta = json.loads((ROOT / "results" / "ml1m_genre_matrix.json").read_text(encoding="utf-8"))

print("Dataset summary:", json.dumps(split_stats["dataset_summary"], indent=2))
print("Slice counts:", json.dumps(slices["counts"], indent=2))
print("Genre matrix meta shape:", genre_meta["shape"])

npz_path = ROOT / "results" / "ml1m_genre_matrix.npz"
if npz_path.exists():
    with np.load(npz_path) as data:
        key = "genre_matrix" if "genre_matrix" in data.files else data.files[0]
        gmat = data[key]
        print(f"Loaded {npz_path.name}[{key!r}]: shape={gmat.shape}, dtype={gmat.dtype}, row0_sum={gmat[0].sum():.1f}")

## 3. Fast Pilot Self-Test (`python/rec`)

Runs the 8-arm NumPy port (`bpr`, `lightgcn`, `side`, `ngcf`, `last`, `loop`, `learned`, `knn`) on the 66-user MovieLens latest-small pilot (12 epochs, seed `2020`). Note: these are pilot sanity numbers, **not** the graded `ml-1m` table.

In [ ]:
# %% Run 8-arm NumPy pilot self-test
from rec.selftest import run
from rec.data import DATA

pilot_results = {
    "bpr": run("bpr", "bpr"),
    "lightgcn": run("light", "lightgcn"),
    "side": run("side", "side"),
    "ngcf": run("ngcf", "ngcf", lr=0.01),
    "last": run("last", "lightgcn", combine="last"),
    "self_loops": run("loop", "lightgcn", self_loops=True),
    "learned": run("learned", "lightgcn", combine="learned"),
    "knn": run("knn", "knn"),
}
print(f"Pilot complete: {len(DATA['users'])} users, {len(DATA['items'])} items.")

## 4. Official `ml-1m` Training (RecBole)

Runs `LightGCN` and baselines under `config/base.yaml` (ratings $\ge 3$, 8:1:1 RS split, full-sort evaluation, `valid_metric: NDCG@10`). Set `RUN_TRAINING = True` on a GPU machine.

In [ ]:
# %% Train LightGCN and baselines on ml-1m across seeds 2020, 2021, 2022
RUN_TRAINING = False  # Flip to True on GPU machine (each 500-epoch LightGCN run takes ~85 min on GTX 1660)

MODELS = ["LightGCN", "BPR", "ItemKNN", "NGCF", "NeuMF", "SASRec"]
SEEDS = [2020, 2021, 2022]

if RUN_TRAINING:
    from recbole.quick_start import run_recbole

    for model_name in MODELS:
        cfg_list = [str(ROOT / "config" / "base.yaml")]
        if model_name == "LightGCN":
            cfg_list.append(str(ROOT / "config" / "lightgcn.yaml"))
        for seed in SEEDS:
            print(f"=== Running {model_name} | seed={seed} ===")
            res = run_recbole(
                model=model_name,
                dataset="ml-1m",
                config_file_list=cfg_list,
                config_dict={"seed": seed, "reproducibility": True},
            )
            print(res)
else:
    official_path = ROOT / "results" / "lightgcn_ml1m_run.json"
    if official_path.exists():
        official = json.loads(official_path.read_text(encoding="utf-8"))
        print("Existing official LightGCN (seed 2020) test result:")
        print(json.dumps(official["result"]["test_result"], indent=2))

## 5. Controlled Extensions (`LightGCNAdaptive` & `LightGCNSide`)

Tests the two single-factor extensions separately on `ml-1m`:
- **`LightGCNAdaptive`**: Softmax-learned layer weights initialized at 0 (uniform at step 0).
- **`LightGCNSide`**: Linear projection of the L1-normalized `[3629, 18]` genre matrix added to $E^{(0)}$.

In [ ]:
# %% Run LightGCNAdaptive and LightGCNSide (set RUN_EXTENSIONS = True on GPU)
RUN_EXTENSIONS = False

if RUN_EXTENSIONS:
    from recbole.quick_start import run_recbole
    from model.lightgcn_adaptive import LightGCNAdaptive
    from model.lightgcn_side import LightGCNSide

    base_cfgs = [str(ROOT / "config" / "base.yaml"), str(ROOT / "config" / "lightgcn.yaml")]

    for seed in [2020, 2021, 2022]:
        print(f"=== LightGCNAdaptive | seed={seed} ===")
        run_recbole(
            model=LightGCNAdaptive,
            dataset="ml-1m",
            config_file_list=base_cfgs,
            config_dict={"seed": seed, "reproducibility": True},
        )

        print(f"=== LightGCNSide | seed={seed} ===")
        run_recbole(
            model=LightGCNSide,
            dataset="ml-1m",
            config_file_list=base_cfgs,
            config_dict={
                "seed": seed,
                "reproducibility": True,
                "genre_matrix_path": str(ROOT / "results" / "ml1m_genre_matrix.npz"),
                "n_genres": 18,
            },
        )

## 6. Cold-Item Slice Scoring & `results/runs.csv` Export

Scores any saved RecBole `.pth` checkpoint on `all`, `cold_item` (372 items, $\le 5$ train edges), `zero_item` (28 items), and `warm_item` (3,228 items) using `scripts/cold_start_report.py`, appending rows to `results/runs.csv`.

In [ ]:
# %% Score saved checkpoint(s) on cold-item slices -> results/runs.csv
from scripts.cold_start_report import append_runs_csv, evaluate_slices

ckpt_path = ROOT / "saved" / "LightGCN-Nov-14-2026_21-18-31.pth"
runs_csv = ROOT / "results" / "runs.csv"

if ckpt_path.exists():
    try:
        meta, summary = evaluate_slices(ckpt_path, ROOT / "results" / "ml1m_cold_item_slices.json")
        append_runs_csv(
            runs_csv,
            meta,
            summary,
            combine="uniform",
            self_loop=False,
            seconds=5103.89,
            gpu_mb=0.0,
            epoch=125,
        )
        print("Appended slice results to", runs_csv)
        print(json.dumps(summary, indent=2))
    except ImportError as exc:
        print("Skipping RecBole checkpoint scoring (missing torch/recbole):", exc)
else:
    print("Checkpoint not found locally:", ckpt_path)